**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

# Come impara una rete
Computer Vision · Lezione 11

**Notebook dimostrativo completo.** Tutto il codice è già scritto; gli esempi e le spiegazioni servono a seguire la lezione e a ripassarla. Gli spunti N01–N12 sono letture guidate con spiegazioni disponibili. 

### Richiamo delle lezioni 9–10
Un **batch** è un gruppo di esempi; uno **step** aggiorna i parametri; un’**epoca** percorre i batch del training. Il **checkpoint del modello** è una copia indipendente dello stato da ripristinare prima del report: è diverso da una sosta didattica. `train()` e `eval()` impostano la modalità dei moduli; `no_grad()` evita di registrare il grafo, senza sostituire `eval()` o autorizzare aggiornamenti nella valutazione. Riprendiamo qui il [ciclo di apprendimento](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) e la [distinzione fra modalità e gradienti](https://docs.pytorch.org/docs/2.14/notes/autograd.html#locally-disabling-gradient-computation) già usati nella Lezione 10. Score `s`, probabilità `p` e cross-entropy mantengono la notazione delle lezioni 9 e 10.

Cella Cxx indica la posizione fisica, contando anche Markdown. Le celle successive sono esempi svolti per una consultazione facoltativa. I dati sono già inclusi nel pacchetto.


Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

## Avvio in Google Colab

Usare un runtime Python con **acceleratore hardware Nessuno**. Gli esempi restano su CPU e float64. Caricare nello stesso runtime il notebook e tutto `Come_impara_una_rete_studenti.zip`; il solo `.ipynb` non trasferisce i dati. Prima di C03 usare una cella temporanea:

Per caricare il pacchetto completo e scegliere la cartella corrente seguire le istruzioni Colab nel README di questo laboratorio. Le celle scientifiche Cxx rimangono invariate.

Controllare `torch`, `numpy`, `matplotlib` e `PIL`. Installare soltanto le librerie mancanti con `%pip install nome_pacchetto` e versioni compatibili; `PIL` è fornito da **Pillow**. JupyterLab non serve in Colab. C03 registra le versioni effettive di PyTorch, NumPy e Python; annotare anche Matplotlib/Pillow quando si confrontano ambienti. Dopo un eventuale riavvio ripetere setup e import. Eliminare la cella temporanea per conservare i **29 ID C01–C29**.

Demo S22: C03, C05, C07, C09, C11 → S23. Dopo C13 i pesi sono cambiati e `.grad` è stato azzerato: per rifare C11 eseguire prima **C09 → C11**. Dopo un nuovo runtime gli output visibili non ricreano le variabili: preparare la cartella e rieseguire i blocchi necessari in ordine.

Per conservare una copia personale della dimostrazione, dopo C29 si può scaricare il `.ipynb` con output da **File → Scarica → Scarica .ipynb** e anche `output_lezione11/` con una cella temporanea:

```python
import shutil
from google.colab import files

shutil.make_archive('/content/output_lezione11', 'zip', root_dir=str(out))
files.download('/content/output_lezione11.zip')
```

Eliminare anche questa cella dopo il download per conservare i 29 ID. C29 esporta soltanto le voci calcolate; il tempo `elapsed_cells_seconds` misura l’intervallo da C03, comprese attese e pause se le celle sono eseguite manualmente, non un tempo CPU o di training. Salvare il notebook non conserva automaticamente i file del runtime. Le verifiche documentate sono locali su CPU; non è stata eseguita una prova nel servizio Colab. [FAQ ufficiale Colab](https://research.google.com/colaboratory/faq.html).

## C02 · Ambiente
Python 3.12, PyTorch 2.14, NumPy, Matplotlib, Pillow. Tutti i calcoli differenziali usano CPU e float64. Il **dtype** è il tipo numerico: float64 indica numeri in virgola mobile a 64 bit, con maggiore precisione di float32. È utile nel controllo delle differenze finite, che sottrae valori vicini. Aprire dalla cartella studenti con dati/ nel livello superiore.

In [ ]:
# C03
import torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
import json, time, platform, hashlib
from IPython.display import display
from PIL import Image
start = time.perf_counter()
torch.set_num_threads(1)
torch.manual_seed(20261019)
torch.set_default_dtype(torch.float64)
plt.rcParams.update({'font.size': 12, 'figure.dpi': 130})
out = Path('output_lezione11'); out.mkdir(exist_ok=True)
roots = [Path.cwd(), *Path.cwd().parents]
data_path = next((p/'dati/digits_cv2026.npz' for p in roots if (p/'dati/digits_cv2026.npz').exists()), None)
assert data_path is not None, 'Aprire il notebook nel pacchetto completo con dati/'
report = {'torch':torch.__version__, 'numpy':np.__version__, 'python':platform.python_version(), 'device':'cpu', 'dtype':'float64', 'seed':20261019, 'data_sha256':hashlib.sha256(data_path.read_bytes()).hexdigest()}
print(report)

## C04 · N01

### Prima del grafo: il neurone
Un neurone calcola una somma pesata e un bias: `a = w*x + b`. `x` è un dato, `w` e `b` sono parametri, `a` è la **preattivazione**. Una funzione di attivazione produce l’uscita `h`: l’identità restituisce `a`, mentre **ReLU** (*Rectified Linear Unit*, unità lineare rettificata) restituisce `max(0, a)`. ReLU lascia passare i valori positivi e azzera i negativi; la sua uscita non è una probabilità.

Il **forward** calcola uscita e loss, cioè la funzione di errore rispetto al target `y`. Il **backward** calcola le sensibilità della loss: il gradiente raccoglie le derivate rispetto ai parametri. Non aggiorna da solo i pesi. Nel grafo seguire prima i valori in avanti, poi un solo percorso di derivate all’indietro.

**Il controllo automatico.** Autograd è il sistema di differenziazione automatica di PyTorch: registra le operazioni sui tensori che richiedono gradienti e, con `backward`, combina le derivate locali del grafo. Calcola derivate, senza aggiornare i pesi. Un **tensore** è un contenitore numerico con una forma: uno **scalare** ha zero assi, un vettore uno e una matrice due. In C05 `x`, `w` e `b` sono tensori scalari.

**Spunto di lettura:** Quali valori produce il forward del neurone e quali gradienti calcola il backward?

**Lettura guidata.** Con `x=2`, `w=3`, `b=−1` e target `y=1`, il forward dà `a=5`, `h=5` e `L=8`. Dal residuo `h−y=4` e dalla derivata della ReLU attiva, uguale a 1, si ottengono `dw=8`, `db=4` e `dx=12`. C05 verifica questi numeri e che i pesi restino invariati: calcolare una derivata rispetto a `x` non rende il dato un parametro appreso.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C05
x = torch.tensor(2., requires_grad=True)
w = torch.tensor(3., requires_grad=True)
b = torch.tensor(-1., requires_grad=True)
a = w*x+b; h = torch.relu(a); loss = (h-1).square()/2
before = [v.detach().clone() for v in (w,b)]
loss.backward()
report['scalar'] = {'a':a.item(),'h':h.item(),'loss':loss.item(),'dx':x.grad.item(),'dw':w.grad.item(),'db':b.grad.item(),'unchanged':all(torch.equal(u,v) for u,v in zip(before,(w,b)))}
assert report['scalar'] == {'a':5.,'h':5.,'loss':8.,'dx':12.,'dw':8.,'db':4.,'unchanged':True}
print(report['scalar'])

## C06 · N02

### Quando lo stesso dato viene riusato
Un arco indica una dipendenza del calcolo. Se una quantità alimenta più ingressi o più operazioni, il reverse raccoglie tutti i percorsi: **moltiplica lungo ogni percorso e somma i contributi che tornano alla stessa quantità**. Questo riuso nello stesso grafo è diverso dall’accumulo nei campi `.grad` fra due chiamate successive a `backward`.

**Spunto di lettura:** Perché i tre contributi che tornano a x si sommano?

**Lettura guidata.** A `x=2` si ha `q=6`. Il prodotto `x*x` usa due volte lo stesso valore e restituisce due contributi pari a 2; il ramo diretto restituisce 1. La derivata totale è quindi `2+2+1=5`, come conferma C07. I rami non corrispondono a tre parametri indipendenti.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C07
xq = torch.tensor(2., requires_grad=True)
q = xq*xq + xq; q.backward()
report['branch'] = {'q':q.item(),'dq_dx':xq.grad.item()}
assert xq.grad.item() == 5.
print(report['branch'])

## C08 · N03

### Che cosa significa MLP
**MLP** significa *Multilayer Perceptron*, percettrone multistrato. Uno strato **denso** (*fully connected*) contiene neuroni che ricevono tutte le uscite dello strato precedente. Qui due input alimentano due neuroni nascosti con ReLU, poi due uscite producono gli **score `s`**, chiamati anche *logits*. I neuroni nascosti producono rappresentazioni intermedie; le uscite finali corrispondono alle classi.

Nello schema, `W1` e `b1` raccolgono pesi e bias nascosti, `z1` le loro preattivazioni, `h` le attivazioni. `W2` e `b2` producono gli score finali `s`; `p` indica le probabilità dopo softmax, come nelle lezioni 9 e 10. Leggere prima una singola somma pesata e poi riconoscerla nel prodotto matriciale. In C09, `hidden` indica le attivazioni nascoste e `scores` gli score finali. La cross-entropy (**CE**) confronta gli score con il target; `delta` indica un gradiente in arrivo, non un nuovo peso.

**Spunto di lettura:** Come si collegano forme, score, cross-entropy e gradienti della MLP fissata?

**Lettura guidata.** I 12 parametri danno preattivazioni e attivazioni nascoste `(1, 2)`, score `s=(−1, 1)` e probabilità circa `(0.119202922, 0.880797078)`. Per il target 0 la CE vale circa `2.126928011`. Se `r=0.880797078`, il gradiente sui due score è `(−r, r)` e quello su `h` è `(−2r, 2r)`. Ogni gradiente di un peso è il prodotto fra il segnale in arrivo e l’ingresso del collegamento; per esempio la prima riga di `dW2` è `(−r, −2r)`. C09 confronta tutti i gradienti con il sistema autograd già introdotto in C04, mantenendo ferme le copie dei parametri iniziali.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C09
x_mlp = torch.tensor([1.,2.])
W1 = torch.eye(2, requires_grad=True); b1 = torch.zeros(2, requires_grad=True)
W2 = torch.tensor([[1.,-1.],[-1.,1.]], requires_grad=True); b2 = torch.zeros(2, requires_grad=True)
params = (W1,b1,W2,b2)
def objective(a,b,c,d):
    hidden = torch.relu(a@x_mlp+b)
    scores = c@hidden+d
    return torch.nn.functional.cross_entropy(scores[None,:], torch.tensor([0]))
initial = [p.detach().clone() for p in params]
mlp_loss = objective(*params); mlp_loss.backward()
r = torch.sigmoid(torch.tensor(2.)).item()
d2 = torch.tensor([-r,r]); d1 = W2.detach().T@d2
analytic = [torch.outer(d1,x_mlp), d1, torch.outer(d2,x_mlp), d2]
report['mlp'] = {'loss':mlp_loss.item(), 'param_count':sum(p.numel() for p in params),'gradients':{k:p.grad.tolist() for k,p in zip(['W1','b1','W2','b2'],params)},'backward_unchanged':all(torch.equal(a,b) for a,b in zip(initial,params)), 'analytic_max_error':max((p.grad-g).abs().max().item() for p,g in zip(params,analytic))}
assert report['mlp']['analytic_max_error'] < 1e-12
print(json.dumps(report['mlp'],indent=2))

## C10 · N04

### Il controllo numerico
La **differenza finita** stima una derivata confrontando due valori della stessa loss con una piccola perturbazione positiva e negativa di un parametro. `fd_step` è questa perturbazione; non è il learning rate di uno step. Un punto angoloso, o *kink*, è un punto dove la derivata classica non esiste, come zero per ReLU. Lo **Jacobiano** raccoglie in una matrice le derivate di più uscite rispetto a più ingressi: compatta le stesse dipendenze del grafo, senza cambiare la regola della catena.

**Spunto di lettura:** Che cosa verifica il confronto fra formula analitica, differenze finite e autograd?

**Lettura guidata.** C11 perturba un parametro alla volta di `fd_step=1e−6` e confronta la differenza centrale con il gradiente già ottenuto in C09. Il controllo richiede gli stessi dati, gli stessi pesi e la stessa loss, senza eseguire uno step. A zero la ReLU non ha derivata classica: la differenza centrale vale `0.5`, mentre autograd adotta `0`. Non è una contraddizione. Il caso rettangolare separa le forme `X: 2×3`, `W: 4×3` e uscita `2×4`, così un errore di trasposta non resta nascosto dal caso quadrato.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C11
fd_step = 1e-6
fd = []
for j,p in enumerate(params):
    g = torch.empty_like(p)
    for k in range(p.numel()):
        plus = [a.clone() for a in initial]; minus = [a.clone() for a in initial]
        plus[j].view(-1)[k] += fd_step; minus[j].view(-1)[k] -= fd_step
        g.view(-1)[k] = (objective(*plus)-objective(*minus))/(2*fd_step)
    fd.append(g)
err = max((p.grad-g).abs().max().item() for p,g in zip(params,fd))
check = torch.autograd.gradcheck(objective,tuple(p.detach().clone().requires_grad_() for p in params),eps=fd_step,atol=1e-6,rtol=1e-4)
z = torch.tensor(0.,requires_grad=True); torch.relu(z).backward()
report['gradcheck']={'fd_step':fd_step,'fd_max_error':err,'gradcheck':check,'relu_zero_autograd':z.grad.item(),'relu_zero_central':.5}
assert err < 1e-6 and check
# Forma rettangolare: B=2, D=3, M=4. La loss media contiene gia 1/B.
X = torch.arange(6.).reshape(2,3)/5
Wr = torch.arange(12.).reshape(4,3).div(12).requires_grad_(); br = torch.zeros(4,requires_grad=True)
Yr = X@Wr.T+br; Yr.retain_grad(); lr = Yr.square().sum(dim=1).mean()/2; lr.backward()
assert torch.allclose(Wr.grad,Yr.grad.T@X) and torch.allclose(br.grad,Yr.grad.sum(0))
report['gradcheck']['rectangular_shapes'] = [list(X.shape),list(Wr.shape),list(Yr.shape)]
print(report['gradcheck'])

## C12 · N05

### Dal gradiente al nuovo parametro
Un **ottimizzatore** usa i gradienti per aggiornare i parametri. Nella versione semplice, **SGD** (*Stochastic Gradient Descent*) sottrae il gradiente del batch moltiplicato per il **learning rate** `eta`: `parametro_nuovo = parametro_precedente - eta*gradiente`. `backward` calcola le derivate, `step` applica l’aggiornamento e `zero_grad` delimita quali contributi devono entrare nello step successivo.

**Spunto di lettura:** Quali quantità cambiano con backward, step e zero_grad?

**Lettura guidata.** `backward` calcola e accumula `.grad`; C09 ha già verificato che non cambia i pesi. C13 applica uno step SGD con learning rate `0.01` e poi pone i gradienti a `None` con `zero_grad`. Per rifare C11 dopo questo passaggio occorre ricreare il caso iniziale con **C09 → C11**. Il reset dei gradienti non ripristina i pesi precedenti.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C13
opt = torch.optim.SGD(params, lr=.01)
old = [p.detach().clone() for p in params]
opt.step()
assert all(torch.allclose(p,oldp-.01*g) for p,oldp,g in zip(params,old,analytic))
opt.zero_grad(set_to_none=True)
assert all(p.grad is None for p in params)
report['step']={'parameters_changed':any(not torch.equal(a,b) for a,b in zip(old,params)), 'grads_none':all(p.grad is None for p in params)}
print(report['step'])

## C14 · N06

### Simboli dell’aggiornamento
`theta` è un parametro, `g` il gradiente corrente e `eta` il learning rate. Un **buffer** è una quantità conservata fra gli aggiornamenti. Il **momentum** mantiene un buffer `v` che combina il gradiente corrente con quelli precedenti secondo la convenzione dichiarata; il passo può quindi avere una direzione diversa da quella suggerita dal solo `g` corrente.

**Spunto di lettura:** Come cambiano il parametro e lo stato con SGD e momentum?

**Lettura guidata.** Sulla quadratica `L(theta)=theta²/2`, da `theta=2` il gradiente è 2: con `eta=0.5` si arriva a `theta=1`, loss `0.5`; con `eta=2.5` a `theta=−3`, loss `4.5`, superiore al valore iniziale 2. Nel secondo esempio i gradienti sono assegnati `(2, −1)`: il momentum `v=0.9*v+g` produce buffer `(2, 0.8)` e parametri `(1.8, 1.72)`. Quel secondo gradiente non viene ricavato dalla quadratica del primo esempio.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C15
sgd = []
for eta in [.5,2.5]:
    theta=torch.tensor(2.,requires_grad=True); optimizer=torch.optim.SGD([theta],lr=eta)
    (theta.square()/2).backward(); optimizer.step()
    sgd.append({'eta':eta,'theta':theta.item(),'loss':theta.item()**2/2})
theta=torch.tensor(2.,requires_grad=True); optimizer=torch.optim.SGD([theta],lr=.1,momentum=.9)
momentum=[]
for grad in [2.,-1.]:
    theta.grad=torch.tensor(grad); optimizer.step()
    momentum.append({'gradient':grad,'buffer':optimizer.state[theta]['momentum_buffer'].item(),'theta':theta.item()})
report['sgd']=sgd; report['momentum']=momentum
print(sgd); print(momentum)

## C16 · N07

### Che cosa conserva Adam
**Adam** significa *Adaptive Moment Estimation*. Mantiene due memorie per ciascun parametro. Una **media esponenziale** combina la memoria precedente e l’osservazione corrente: `nuova_memoria = beta*memoria + (1−beta)*osservazione`. Le osservazioni più lontane ricevono pesi che decadono come potenze di `beta`.

Adam usa `m_t = beta1*m_(t−1) + (1−beta1)*g_t` per i gradienti e `v_t = beta2*v_(t−1) + (1−beta2)*g_t²` per i loro quadrati, partendo da `m_0 = v_0 = 0`. Qui `v` conserva quadrati: **non è il buffer `v` del momentum** in C14. Il primo momento riguarda la media; il secondo momento è la media dei quadrati. Non è la varianza centrata, che richiederebbe anche la sottrazione del quadrato della media.

Le quantità `m_hat = m_t/(1−beta1**t)` e `v_hat = v_t/(1−beta2**t)` correggono il fattore iniziale dovuto alle memorie nulle: il *bias* di questa correzione è diverso dal **bias del neurone**, il parametro additivo `b`. Lo stato ricorrente resta `m, v`; le quantità con il cappello si ricavano per lo step.

`epsilon_Adam` è una piccola costante che stabilizza il denominatore `sqrt(v_hat) + epsilon_Adam`, **fuori dalla radice**. Non è `epsilon_FD = fd_step`, la perturbazione delle differenze finite di C10, e non è il learning rate `eta`. Nel caso seguente `beta1=0.9`, `beta2=0.999` e `epsilon_Adam=1e−8`.

**Spunto di lettura:** Che cosa corregge il primo passo Adam e perché il secondo momento non è una varianza?

**Lettura guidata.** Con gradiente 2, C17 produce `m=0.2` e `v=0.004`. Le correzioni iniziali danno `m_hat=2` e `v_hat=4`; con learning rate `0.1` ed epsilon `1e−8` fuori dalla radice si ottiene `theta≈1.9000000005`. `v_hat` stima il secondo momento: se il gradiente fosse sempre 2, il suo quadrato medio sarebbe 4 e la sua varianza sarebbe invece 0.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C17
theta=torch.tensor(2.,requires_grad=True)
optimizer=torch.optim.Adam([theta],lr=.1,betas=(.9,.999),eps=1e-8,weight_decay=0)
theta.grad=torch.tensor(2.); optimizer.step()
st=optimizer.state[theta]
report['adam']={'theta':theta.item(),'m':st['exp_avg'].item(),'v':st['exp_avg_sq'].item(),'step':st['step'].item()}
assert abs(theta.item()-1.9000000005)<1e-12
print(report['adam'])

## C18 · N08
### Prima delle scale: inizializzazione e connessioni
L’**inizializzazione** assegna i valori dei parametri prima degli aggiornamenti. **Xavier**, chiamata anche **Glorot**, e **He** sono regole per scegliere la scala iniziale dei pesi in rapporto alle connessioni e all’attivazione. Non sono risultati di un addestramento.

Per uno strato denso **3 → 2**, ogni neurone di uscita riceve 3 ingressi: `fan_in=3`; le uscite dello strato sono 2: `fan_out=2`. La matrice dei pesi ha forma `2×3`. Nel caso numerico **64 → 32**, `fan_in=64`, `fan_out=32` e `W` ha forma `32×64`. Questi conteggi non sono la dimensione del batch. `gain` è un fattore moltiplicativo della deviazione standard; la varianza ne riceve il quadrato. Qui Xavier usa `gain=1`; He è riferita alla ReLU e a `fan_in`.

La **varianza** misura la dispersione quadratica e la **deviazione standard** `sigma` ne è la radice. Una **normale centrata** ha densità a campana e code non limitate, con varianza `sigma²`. Una **uniforme centrata** ha densità costante tra `−a` e `a`, nessun valore fuori dall’intervallo e varianza `a²/3`: `a` è un limite, non la deviazione standard. Stessa varianza non significa stessa distribuzione. C19 calcola le scale teoriche e un gradiente del modello lineare a zero: non estrae nuovi campioni Xavier/He e non esegue un training.

**Spunto di lettura:** Quali scale iniziali producono Xavier e He e quando lo zero è un problema?

**Lettura guidata.** Per lo strato `64→32`, Xavier con gain 1 usa varianza `2/(64+32)=1/48` e deviazione standard circa `0.144337567`; He per ReLU usa varianza `2/64=1/32` e deviazione circa `0.176776695`. C19 mostra anche un classificatore lineare a pesi e bias nulli con gradiente non nullo. Questo non giustifica una MLP ReLU interamente nulla: unità nascoste simmetriche possono restare indistinguibili. Bias zero e pesi distinti sono compatibili.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C19
report['initialization']={'xavier_variance':2/96,'xavier_std':(2/96)**.5,'he_variance':2/64,'he_std':(2/64)**.5}
# Linear a zero puo avere un gradiente non nullo.
linear=torch.nn.Linear(2,2); torch.nn.init.zeros_(linear.weight);torch.nn.init.zeros_(linear.bias)
ce=torch.nn.functional.cross_entropy(linear(torch.tensor([[1.,2.]])),torch.tensor([0]));ce.backward()
report['initialization']['linear_zero_grad']=linear.weight.grad.tolist()
print(report['initialization'])

## C20 · N09
**Spunto di lettura:** Quando L2 e decay separato coincidono e perché Adam richiede una distinzione?

**Lettura guidata.** Nel caso SGD semplice di C21, il gradiente totale è `3+0.1*2=3.2` e il nuovo peso è `2−0.05*3.2=1.84`. La forma separata `(1−0.05*0.1)*2−0.05*3` dà lo stesso valore. Con Adam accoppiato la penalità entra nei momenti; con AdamW la contrazione resta separata. I primi passi del caso fissato danno circa `1.95` e `1.94`: il confronto illustra meccanismi diversi e non misura quale metodo sia migliore su immagini.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C21
decay={}
for name,cls,kw in [('SGD_L2',torch.optim.SGD,{}),('Adam_L2',torch.optim.Adam,{'decoupled_weight_decay':False}),('AdamW',torch.optim.AdamW,{})]:
    weight=torch.tensor(2.,requires_grad=True)
    optimizer=cls([weight],lr=.05,weight_decay=.1,**kw)
    weight.grad=torch.tensor(3.);optimizer.step();decay[name]=weight.item()
report['decay']=decay
assert abs(decay['SGD_L2']-1.84)<1e-12
assert abs(decay['Adam_L2']-decay['AdamW'])>.009
print(decay)

## C22 · N10
**Spunto di lettura:** Quali uscite produce il dropout in train e in eval e che cosa conserva il valore atteso?

**Prima del calcolo: che cos’è il dropout?** Durante il training, il dropout azzera temporaneamente alcune attivazioni tramite una maschera casuale. Il neurone e i suoi pesi restano nella rete: una nuova maschera può mantenere un’attivazione prima azzerata. Lo scopo è regolarizzare il modello, limitando la dipendenza sistematica da poche attivazioni; un vantaggio sulle metriche non è garantito. `p_drop` è la probabilità di azzerare; `q_keep = 1 − p_drop` è la probabilità di mantenere, non la stessa quantità. Nelle formule delle figure `p = p_drop` e `q = q_keep`.

**Poi lo scaling: inverted dropout.** In training ogni elemento mantenuto è diviso per `q_keep`: `u = m · h / q_keep`, con `m` pari a 1 con probabilità `q_keep` e a 0 con probabilità `p_drop`. Per questa formula si assume `0 ≤ p_drop < 1`. Questo conserva `E[u] = h` per il singolo elemento. In eval il layer è l’identità: `u = h`, senza maschere e senza ulteriore scaling. Il ragionamento segue lo stesso ordine: maschera, scaling, limite non lineare.

**Lettura guidata.** In training, l’ingresso 2 diventa 0 con probabilità `0.25` oppure `8/3` con probabilità `0.75`; il valore atteso resta 2. In eval il layer restituisce 2. Il campione di 12 elementi può non annullarne esattamente un quarto. Per `f(u)=ReLU(u−1)`, `f(E[u])=1` ma `E[f(u)]=1.25`: conservare la media all’uscita del dropout non equivale a conservare la predizione media dopo altri strati non lineari.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C23
prob=np.array([.25,.75]); outcomes=np.array([0.,8/3])
drop=torch.nn.Dropout(.25);drop.train();torch.manual_seed(20261019)
sample=drop(torch.full((12,),2.));drop.eval(); eval_output=drop(torch.tensor([2.]))
report['dropout']={'outcomes':outcomes.tolist(),'probabilities':prob.tolist(),'expectation':float(prob@outcomes),'f_expectation':1.,'expectation_f':float(prob@np.maximum(outcomes-1,0)),'sample12':sample.tolist(),'eval':eval_output.item()}
print(report['dropout'])

## C24 · N11
**Prima delle immagini.** La **data augmentation** genera versioni trasformate degli esempi di training. Le trasformazioni **geometriche** cambiano posizioni o forme, come flip e rotazioni; quelle **fotometriche** cambiano le intensità, per esempio luminosità o contrasto.

**Spunto di lettura:** Che cosa mostrano le trasformazioni delle cifre e come si controlla il significato del target?

**Lettura guidata.** C25 mostra esempi training delle classi 2, 5, 6 e 9 con originale, flip orizzontale, rotazione di 180° e traslazione di un pixel a destra con riempimento 0, sempre nella scala `0–16`. Una trasformazione può cambiare i tratti che giustificano l’etichetta: per esempio una rotazione ampia può rendere un 6 simile a un 9. La griglia serve a discutere questa validità, non misura un vantaggio di accuracy. In segmentazione la stessa geometria va applicata alla maschera delle classi. Se serve interpolazione, il **vicino più prossimo** (*nearest neighbor*) prende il codice del pixel di ingresso più vicino, senza inventare etichette intermedie. Per i box vanno trasformate e controllate anche le coordinate.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C25
data=np.load(data_path)
print('Campi dati:',data.files)
images=data['images']; targets=data['target']; train=data['train_idx']
ids=[int(next(i for i in train if targets[i]==label)) for label in [2,5,6,9]]
fig,axes=plt.subplots(4,4,figsize=(9.6,8),layout='constrained')
for row,idx in enumerate(ids):
    original=images[idx]; shifted=np.zeros_like(original); shifted[:,1:]=original[:,:-1]
    variants=[original,np.fliplr(original),np.rot90(original,2),shifted]
    for col,img in enumerate(variants):
        ax=axes[row,col];ax.imshow(img,cmap='gray_r',vmin=0,vmax=16,interpolation='nearest');ax.set_xticks([]);ax.set_yticks([])
        if row==0:ax.set_title(['Originale','Flip orizzontale','Rotazione 180°','Sposta +1 pixel x'][col])
        if col==0:ax.set_ylabel(f'ID {idx}, classe {targets[idx]}')
fig.savefig(out/'digits_transformations.png',dpi=180);plt.show()
report['transformations']={'training_ids':ids,'labels':[int(targets[i]) for i in ids],'range':[0,16],'interpolation':'nessuna, permutazioni intere','shift_fill':0,'axis_x':'colonne verso destra'}

## C26 · N12
### Prima del confronto: i nomi dei componenti
**CNN** significa *Convolutional Neural Network*, rete neurale convoluzionale: il modello studiato nella Lezione 12. Un’**ablation** è un confronto controllato in cui si modifica o rimuove un solo componente mantenendo fissate le altre condizioni.

**BatchNorm**, o **BN** (*Batch Normalization*), è una normalizzazione delle attivazioni basata sulle statistiche del mini-batch durante il training e su statistiche conservate per l’eval. Ha quindi anche uno **stato**, oltre ai parametri appresi. **Identity** è il passaggio invariato `uscita = ingresso`. Qui identifichiamo soltanto i due componenti da confrontare; il funzionamento numerico e la gestione dello stato sono sviluppati nella Lezione 12.

**Spunto di lettura:** Qual è il fattore variato nell’ablation comune dello Studio 12 e quali condizioni restano fisse?

**Lettura guidata.** C27 stampa un protocollo già predisposto: l’unico fattore è **BatchNorm rispetto a Identity** nella CNN dello Studio 12. Restano fissi split, pesi comuni iniziali, ordine dei batch, Adam, budget di 30 epoche e criteri di validation. L’ipotesi precede la prova; la voce `da misurare nello Studio` non è un risultato sperimentale. Qui si legge il disegno del confronto: non occorre compilare una nuova proposta né eseguire un secondo esperimento nella Lezione 11.

La cella seguente è completa: si può leggerne il codice o eseguirla in ordine per confrontare l’output con la spiegazione.

In [ ]:
# C27
protocol = {'baseline':'CNN dello Studio 12','fattore_assegnato':'BatchNorm rispetto a Identity','invarianti':['split','pesi comuni iniziali','ordine dei batch','Adam','budget 30 epoche','criteri validation'],'risultato':'da misurare nello Studio'}
print(json.dumps(protocol,ensure_ascii=False,indent=2))

## C28 · Registro dei controlli
Il tempo stampato misura questa esecuzione, non il tempo didattico della demo.

In [ ]:
# C29
report['elapsed_cells_seconds']=time.perf_counter()-start
(out/'risultati_verificati.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
print('Controlli completati. Risultati salvati in',out/'risultati_verificati.json')